![RAG Evaluation](./images/RAG-Evaluation.png)


We will be evaluating Context relevance, Groundedness(Faithfullness),Correctenss(Answer relevance)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_ibm import ChatWatsonx

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

MODEL_ID = os.getenv('WATSONX_MODEL', "meta-llama/llama-3-3-70b-instruct") #Used for evaluation
WATSONX_APIKEY = os.getenv('WATSONX_APIKEY')
WATSONX_PROJECT_ID = os.getenv('WATSONX_PROJECT_ID')
WATSONX_URL = os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")

/Users/janibasha/MyProjects/Generative-AI/LLM_Evaluation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=20
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings
)
# vectorstore = Chroma.from_documents(
#         documents=pages_split,
#         embedding=embeddings,
#         persist_directory=persist_directory,
#         collection_name=collection_name
#     )

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6232.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
retriever.invoke("what is agents")

[Document(id='05009e91-b0a7-4018-869e-b7dfe10e5625', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [4]:
MODEL_PARAMETERS= {
    "temperature": 0.5,
    "max_tokens": 500,
    "top_p": 0.9,
    "top_k": 50,
    "stop_sequences": ["\n"]
}
 
def create_llm_client(model):
    """Create and return a WatsonX LLM client"""
    llm = ChatWatsonx(
        model_id=model,
        url=WATSONX_URL,
        apikey=WATSONX_APIKEY,
        project_id=WATSONX_PROJECT_ID,
        params=MODEL_PARAMETERS
    )
    return llm


def generate_text(llm, prompt: str):
    """Generate text using the LLM"""
    response = llm.invoke(prompt)
    return response


In [5]:
llm = create_llm_client("meta-llama/llama-3-3-70b-instruct")
llm

ChatWatsonx(model_id='meta-llama/llama-3-3-70b-instruct', project_id='218eb1d9-5b5c-41da-af21-35a72012e8b6', url=SecretStr('**********'), apikey=SecretStr('**********'), api_key=SecretStr('**********'), params={'temperature': 0.5, 'max_tokens': 500, 'top_p': 0.9, 'top_k': 50, 'stop_sequences': ['\n']}, watsonx_model=<ibm_watsonx_ai.foundation_models.inference.model_inference.ModelInference object at 0x16ac73f90>)

In [6]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [7]:
rag_bot("What is agents")

{'answer': 'In the context of the provided source, an agent refers to a LLM-powered autonomous entity that can perform tasks, interact with its environment, and make decisions. These agents are designed to behave conditioned on past experience and can interact with other agents. They are powered by Large Language Models (LLMs) and combine mechanisms such as planning, memory, and reflection to enable their behavior.',
 'documents': [Document(id='05009e91-b0a7-4018-869e-b7dfe10e5625', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Ove

### Dataset

In [8]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)


{'example_ids': ['54e43f89-6182-4d82-bf35-082cd19caf35',
  '18557b23-921a-4f76-8064-9ad6d705ef3a',
  'd12cebeb-bc99-406c-ae85-26c692b98a62'],
 'count': 3,
 'as_of': '2026-03-07T14:19:52.725832543Z'}

In [ ]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

grader_llm = create_llm_client(model="meta-llama/llama-3-3-70b-instruct").with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
    QUESTION: {inputs['question']}
    GROUND TRUTH ANSWER: {reference_outputs['answer']}
    STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    print("Correctness::::", grade)
    return grade["correct"]

In [18]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = create_llm_client(model="meta-llama/llama-3-3-70b-instruct").with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    print("Answer Relevance::: ", grade)
    return grade["relevant"]

In [19]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = create_llm_client(model="meta-llama/llama-3-3-70b-instruct").with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])

    print("Groundedness::: ", grade)
    return grade["grounded"]

In [20]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = create_llm_client(model="meta-llama/llama-3-3-70b-instruct").with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    print("retrieval::: ", grade)
    return grade["relevant"]

In [21]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-f603a1b2' at:
https://smith.langchain.com/o/5160b42a-75e3-4626-b3e7-d06fe076483e/datasets/03aa1ef0-2119-4ebb-aeb9-a6dfbaa3634c/compare?selectedSessions=f5de3dc9-2c31-48bd-b8d1-6d36a6cced5f




0it [00:00, ?it/s]

grade:::: {'correct': True, 'explanation': "To determine the correctness of the student's answer, let's break down the grading criteria and compare the student's answer to the ground truth answer. (1) The student answer must be factually accurate relative to the ground truth answer. The student lists the same three biases as the ground truth answer: Majority label bias, Recency bias, and Common token bias. This indicates that the student's answer is factually accurate. (2) The student answer must not contain any conflicting statements. The additional information provided by the student about the potential impact of these biases on model performance does not conflict with the ground truth answer. (3) It is acceptable if the student answer contains more information than the ground truth answer, as long as it is factually accurate. The student's answer includes extra details about the potential effects of these biases, but this does not contradict the ground truth. Therefore, the student'

1it [00:21, 21.75s/it]

retrieval:::  {'explanation': "To determine the relevance of the provided facts to the question, we need to analyze the content of the facts and identify any keywords or semantic meaning related to the question. The question asks about the types of biases that can arise with few-shot prompting. The facts provided contain a section titled 'Many studies looked into how to construct in-context examples to maximize the performance' which mentions few-shot classification and proposes that several biases with LLM contribute to high variance. Specifically, it mentions three biases: (1) Majority label bias, (2) Recency bias, and (3) Common token bias. These biases are directly related to few-shot prompting, which is the topic of the question. Although the facts contain a lot of additional information that may not be directly related to the question, the presence of keywords such as 'few-shot', 'biases', and 'LLM' indicates that the facts are relevant to the question. Furthermore, the facts pro

...

retrieval:::  {'explanation': 'To answer this question, we need to analyze the provided facts and identify the parts that are relevant to the ReAct agent and self-reflection.', 'relevant': True}
grade:::: {'correct': True, 'explanation': "To determine the correctness of the student's answer, let's break it down step by step relative to the ground truth answer and the given criteria for grading.\n1. **Factual Accuracy**: The student answer lists the same five types of adversarial attacks as the ground truth answer: (1) Token manipulation, (2) Gradient-based attacks, (3) Jailbreak prompting, (4) Humans in the Loop Red-teaming (which corresponds to Human red-teaming in the ground truth), and (5) Model Red-teaming. The slight variation in wording between 'Human red-teaming' and 'Humans in the Loop Red-teaming' can be considered semantically equivalent in this context, as both refer to the involvement of humans in testing or attacking a system.\n2. **Conflicting Statements**: The student an

3it [01:10, 23.90s/it]

retrieval:::  {'explanation': "To determine the relevance of the provided FACTS to the QUESTION, we need to analyze the content of both. The QUESTION asks about the five types of adversarial attacks. The FACTS provided discuss various aspects of adversarial attacks on Large Language Models (LLMs), including defense mechanisms, types of attacks, and mitigation strategies. Specifically, the section 'Types of Adversarial Attacks#' in the FACTS mentions presenting five approaches to find adversarial inputs, which directly relates to the QUESTION about the types of adversarial attacks. Although the FACTS do not explicitly list the five types in the provided snippet, the mention of 'five approaches' and the subsequent listing of specific attack methods (Token Manipulation, Gradient-based Attacks, Jailbreak Prompting, Humans in the Loop Red-teaming, Model Red-teaming) in the table of contents implies that the FACTS contain relevant information about the types of adversarial attacks. Therefore

3it [01:11, 23.70s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id,feedback.wrapper
0,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), three types o...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,False,True,True,2.200665,18557b23-921a-4f76-8064-9ad6d705ef3a,019cc8b8-80c4-7d13-a5e8-84c295804421,NaN
1,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection to improv...,[page_content='Self-Reflection#\nSelf-reflecti...,None,"ReAct integrates reasoning and acting, perform...",False,True,NaN,True,1.994393,54e43f89-6182-4d82-bf35-082cd19caf35,019cc8b8-d5bc-7040-a554-421cacc97261,NaN
2,What are five types of adversarial attacks?,The source document lists five approaches to f...,[page_content='One simple and intuitive way to...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,2.015186,d12cebeb-bc99-406c-ae85-26c692b98a62,019cc8b9-3316-7ea2-b4f7-20c66a122298,NaN


![](./images/Langsmith-exp-eval.png)